# Prueba Matías Fernández

Implementación de heurística simple que elige los vehículos más cercanos factibles para cada trip. Para problema de recoger pasajeros en The Optimal

## Lógica de la Heurística

El algoritmo busca asignar iterativamente un conjunto de viajes a una flota de vehículos minimizando principalmente los kilómetros recorridos a buscar pasajeros y cumpliendo estrictamente con todas las restricciones del problema, en caso de que no se pueda, simplemente dejo a los trips sin asignar y los reporto en los outputs

### Pasos del algoritmo:

1. **Ordenamiento Inicial:** 
   Se ordenan todos los viajes cronológicamente de forma ascendente según su `hora_presentacion`. Esto prioriza la atención de los pasajeros que deben viajar más temprano.

2. **Evaluación de Inserción (Greedy):**
   Iteramos viaje por viaje. Para cada viaje, evaluamos todos los vehículos disponibles y simulamos su inserción al final de la ruta actual del vehículo. Durante la simulación verificamos:
   - **Capacidad:** Que el número de pasajeros no exceda la capacidad máxima del vehículo.
   - **Tiempos y distancias:** Calculamos el tiempo de viaje desde la ubicación actual del vehículo hasta el origen del viaje utilizando la **Fórmula de Haversine** y asumiendo una **velocidad constante de 20 km/h**.
   - **Ventanas de tiempo (Espera):** Si el vehículo llega al origen antes de la `hora_presentacion`, debe esperar. El tiempo de "pickup" real es el máximo entre la hora de llegada del vehículo y la hora de presentación exigida.
   - **Fin de Jornada:** Simulamos el traslado completo (desde el origen hasta el destino del viaje) y el retorno a la base final del vehículo (`fin_jornada`). Si el tiempo calculado de retorno a la base excede la hora de término de la jornada del vehículo, **la inserción se rechaza**.

3. **Función de Costo:**
   Para los vehículos factibles, se calcula un "costo" de asignación:
   `Costo = Distancia en vacío (Deadhead al origen)`
   *Nota: Se prioriza estrictamente la minimización de kilómetros muertos para asignar el viaje al vehículo más cercano en términos de distancia.*

4. **Asignación y Actualización:**
   El viaje se asigna al vehículo con el **menor costo factible**. Acto seguido, se actualiza el estado del vehículo (nueva ubicación actual, tiempo actual, acumulación de kilómetros muertos) y se registran los nodos de `pickup` y `dropoff` en su ruta. Los viajes que no logran cumplir las restricciones con ningún vehículo se marcan como `unassigned`.

## Estructura del Proyecto

El código está desarrollado aplicando Programación Orientada a Objetos (POO) y su estructura principal consta de los siguientes archivos y directorios:

- `models.py`: Define las entidades del dominio.
  - Función `haversine` para cálculo de distancias.
  - Clase `Location` para representar coordenadas.
  - Clase `Trip` para los viajes y sus cálculos de tiempo de traslado.
  - Clase `Vehicle` que mantiene el estado actual (ubicación, tiempo, capacidad, ruta trazada).

- `heuristic.py`: Contiene la clase `GreedyInsertionHeuristic`, responsable de ejecutar la lógica principal detallada anteriormente.

- `solve.py`: Es el script principal. Lee la instancia JSON, construye los objetos, llama al solver, formatea el JSON de salida y agrega un registro al archivo `metrics.csv`.

- `validator.py`: Script secundario para auditar una solución generada. Verifica la congruencia de los tiempos de viaje (distancias a 20 km/h), que las capacidades no sean superadas, y que se respete el fin de jornada y las horas de presentación.

- `metrics.csv`: Archivo generado automáticamente donde se van acumulando los KPIs de cada ejecución de `solve.py`.
- Directorio `outputs/`: Carpeta generada automáticamente donde se guardan todas las soluciones JSON (para mantener el repositorio ordenado).


## Instrucciones de Uso

### 1. Generar una solución
Ejecutar `solve.py` proporcionando el archivo de entrada, el nombre del archivo de salida deseado, y el archivo de métricas.

Usaré de ejemplo la small, pero simplemente después hay que cambiar el nombre de la instancia por la que se desea revisar

```bash
python solve.py --input instancias/small.json --output small_solution.json --metrics metrics.csv
```
*Este comando procesará la instancia, creará/actualizará el archivo `metrics.csv` añadiendo una fila con los KPIs de la solución, y guardará la estructura del ruteo en `outputs`.*

### 2. Validar una solución
Para correr el auditor de reglas de negocio sobre el JSON generado:

```bash
python validator.py --instance instancias/small.json --solution small_solution.json
```

## Métricas Registradas (`metrics.csv`)
El archivo CSV generado contiene las siguientes columnas:
- `instance_name`: Nombre del archivo procesado.
- `total_trips`: Total de viajes en la instancia.
- `assigned_trips`: Cantidad de viajes exitosamente ruteados.
- `unassigned_trips`: Viajes rechazados por violar las restricciones (capacidad, tiempo o fin de turno).
- `vehicles_used`: Cantidad de vehículos de la flota utilizados.
- `total_deadhead_km`: Sumatoria de todos los kilómetros recorridos por la flota en reposicionamientos sin pasajeros a bordo.
- `total_idle_time_sec`: Tiempo ocioso total (en segundos) que los vehículos pasaron esperando entre el fin de un reposicionamiento y el inicio de un viaje.

In [4]:
import json
import os
from models import Vehicle, Trip


In [11]:
ruta_instancia = "instancias/small.json"
    
with open(ruta_instancia, 'r', encoding='utf-8') as f:
    data = json.load(f)
        
trips = [Trip(t['id'], t['origen'], t['destino'], t['hora_presentacion'], t['num_pasajeros']) for t in data.get('viajes', [])]
vehicles = [Vehicle(v['id'], v['capacidad'], v['inicio_jornada'], v['fin_jornada']) for v in data.get('vehiculos', [])]

vehicles
    

[<models.Vehicle at 0x21d2f2ce200>, <models.Vehicle at 0x21d2f2ce410>]

In [1]:
import numpy as numpy

In [13]:
vehicles[0].capacidad


4

In [ ]:
def solve(self):
        # Iteramos uno por uno sobre la lista de viajes (ya ordenada por hora)
        for trip in self.trips: # iteramos sobre cada viaje
            best_vehicle = None  # Almacenará el mejor vehículo encontrado para este viaje
            best_cost = float('inf')  # Costo inicial infinito. de ahí vamos agregando el vehículo más cercano con la menor distancia
            best_insertion = None      # Almacenará los tiempos simulados (recogida y entrega)

            # Evaluamos todos los vehículos para ver cuál es el elegido por la heurística
            for v in self.vehicles:
                
                # Capacidad.
                # Si el viaje tiene más pasajeros de los que el vehículo soporta, lo ignoramos.
                if trip.num_pasajeros > v.capacidad:
                    continue

                # El vehículo viaja vacío a recoger al pasajero.
                # Distancia desde donde está el vehículo hasta el origen del pasajero.
                dist_to_origin = v.current_location.distance_to(trip.origin)
                # Tiempo que tarda en recorrer esa distancia vacío.
                time_to_origin = v.calculate_travel_time(dist_to_origin)
                # Hora a la que el vehículo llega físicamente al origen del viaje.
                arrival_at_origin = v.current_time + time_to_origin
                
                # Ventana de presentación.
                # Si el vehículo llega antes de tiempo, debe esperar. Se sube al pasajero en la hora mayor.
                pickup_time = max(arrival_at_origin, trip.hora_presentacion)
                
                # El traslado del pasajero.
                # La hora a la que termina el viaje es la hora de recogida + lo que tarda el traslado.
                dropoff_time = pickup_time + trip.travel_time_sec
                
                # Retorno a la base.
                # Simulamos que después de este viaje el vehículo vuelve a casa para ver si el tiempo le alcanza.
                dist_to_base = trip.dest.distance_to(v.base_end)
                time_to_base = v.calculate_travel_time(dist_to_base)
                arrival_at_base = dropoff_time + time_to_base
                
                # Fin de jornada.
                # El vehículo solo es válido si logra volver a la base antes o justo a su hora límite de fin_jornada. 
                if arrival_at_base <= v.fin_jornada:
                    
                    # FUNCIÓN DE COSTO:
                    # Queremos el algoritmo lo más simple posible. El "costo" es solo la 
                    # distancia en vacío. Ganará el vehículo que tenga que viajar menos para recogerlo.
                    cost = dist_to_origin
                    
                    # Si este costo es menor que el mejor encontrado hasta ahora, actualizamos al ganador.
                    if cost < best_cost:
                        best_cost = cost
                        best_vehicle = v
                        best_insertion = {
                            'pickup_time': pickup_time,
                            'dropoff_time': dropoff_time,
                            'deadhead': dist_to_origin,
                            'idle_time': pickup_time - arrival_at_origin
                        }

            # Si encontramos un vehículo que cumplió todo y tiene el menor costo:
            if best_vehicle:
                ins = best_insertion
                
                # Actualizamos el estado del vehículo para prepararlo para su siguiente destino:
                best_vehicle.is_used = True                           # Marcamos que el vehículo salió a trabajar
                best_vehicle.deadhead_km += ins['deadhead']           # Acumulamos los kilómetros que recorrió vacío
                best_vehicle.idle_time_sec += ins['idle_time']        # Acumulamos el tiempo ocioso
                best_vehicle.current_location = trip.dest             # Ahora se encuentra estacionado en el destino
                best_vehicle.current_time = ins['dropoff_time']       # El reloj del vehículo avanza hasta el fin del viaje
                
                # Guardamos un resumen para el Output final que pide el ejercicio
                best_vehicle.assigned_trips.append({
                    'trip_id': trip.id,
                    'pickup_time': round(ins['pickup_time'], 2),
                    'dropoff_time': round(ins['dropoff_time'], 2)
                })
                
                # Agregamos la parada de recogida (pickup) al registro de su ruta
                best_vehicle.route.append({"type": "pickup", "trip_id": trip.id, "lat": trip.origin.lat, "lon": trip.origin.lon, "time": round(ins['pickup_time'], 2)})
                
                # Agregamos la parada de entrega (dropoff) al registro de su ruta
                best_vehicle.route.append({"type": "dropoff", "trip_id": trip.id, "lat": trip.dest.lat, "lon": trip.dest.lon, "time": round(ins['dropoff_time'], 2)})
            else:
                # Si ningún vehículo pudo atender este viaje (por falta de capacidad o falta de tiempo en sus jornadas).
                self.unassigned_trips.append(trip.id)
                
        # Retornamos la lista final con los IDs de los viajes que tristemente no pudieron rutease.
        return self.unassigned_trips